# Tracing & Observability

*Notebook 23*

The OpenAI Agents SDK traces agent runs by default.

This notebook shows you how to read those traces and what to look for when debugging.

You'll also use them to monitor production agents.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, RunConfig, function_tool

# Notebook-specific imports
import time

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"


print("✅ Ready!")

---

## 🔭 What Tracing Captures

Tracing is enabled by default.

A top-level `Runner.run()` creates a trace unless tracing is disabled.

A run inside an active trace joins that trace instead.

A trace is a tree of spans, with a span for each of:

- The run itself, and each agent that handled it

- Each model turn, with its input and response

- Each tool call: name, arguments, and return value

- Handoffs between agents (patterns from Lesson 12)

- Guardrail checks: the guardrail name and whether it tripped (Lesson 20)

- Timing on every span, token usage where reported, and any errors

Payloads (model input, output, and tool data) are captured only when sensitive-data capture is on, the default.

⚠️ **Security note:** Traces can contain prompts, outputs, and tool data.

Protect trace access, and disable payload capture with `RunConfig(trace_include_sensitive_data=False)` when needed.

---

## 🔬 Part 1: Generating Traces Worth Reading

We need an agent with enough structure to make the trace interesting.

In [ ]:
INVENTORY = {
    "laptop": {"price": 1299.99, "stock": 5, "category": "electronics"},
    "headphones": {"price": 149.99, "stock": 23, "category": "electronics"},
    "notebook": {"price": 4.99, "stock": 200, "category": "stationery"},
    "desk lamp": {"price": 39.99, "stock": 0, "category": "electronics"}
}


@function_tool
def search_inventory(query: str) -> str:
    """Search inventory for products matching a query string."""
    matches = [
        f"{name}: ${item['price']:.2f}, category: {item.get('category', 'general')}"
        for name, item in INVENTORY.items()
        if query.lower() in name.lower() or query.lower() in item["category"]
    ]
    return "\n".join(matches) if matches else f"No products found for '{query}'"


@function_tool
def check_stock(product_name: str) -> str:
    """Check the exact stock quantity for a specific named product."""
    key = product_name.lower()
    product = INVENTORY.get(key)
    if not product and key.endswith("s"):
        product = INVENTORY.get(key[:-1])
        if product:
            key = key[:-1]
    if not product:
        return f"Product '{product_name}' not found in inventory"
    status = "In stock" if product["stock"] > 0 else "Out of stock"
    return f"{key}: {product['stock']} units, {status}"


@function_tool
def get_category_report(category: str) -> str:
    """Get a full inventory report for a product category."""
    items = [
        (name, item) for name, item in INVENTORY.items()
        if item["category"] == category.lower()
    ]
    if not items:
        return f"No products in category '{category}'"
    total_value = sum(item["price"] * item["stock"] for _, item in items)
    lines = [f"  {name}: ${item['price']:.2f} × {item['stock']} units" for name, item in items]
    return (
        f"Category: {category}\n"
        + "\n".join(lines)
        + f"\nTotal inventory value: ${total_value:.2f}"
    )


inventory_instructions = (
    "You are an inventory management assistant.\n"
    "- Use search_inventory to find products by name or category\n"
    "- Use check_stock to get exact stock quantities for specific products\n"
    "- Use get_category_report for full category summaries\n"
    "Always use the most specific tool available for each question."
)

inventory_agent = Agent(
    name="InventoryAssistant",
    instructions=inventory_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Inventory agent ready")
print("    Tools: search_inventory | check_stock | get_category_report")


# Helper: the tools a run actually called (from the run items, the same data the trace shows)
def tools_used(result):
    return [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"]

Open the OpenAI traces dashboard at `https://platform.openai.com/traces` in a separate tab.

Your runs appear there under the workflow name each demo sets, like "Run 1: focused stock query".

Export is batched, so a trace may take a moment to show up.

Keep this tab open as you work through the lesson.

In [ ]:
print("🔬 RUN 1: Focused stock question")
print("=" * 60)

start = time.time()

result = await Runner.run(
    inventory_agent,
    input="How many laptops do we have in stock right now?",
    run_config=RunConfig(workflow_name="Run 1: focused stock query"),
)

elapsed = time.time() - start

print(f"Response: {result.final_output}")
print(f"🔧 Tools this run called: {tools_used(result)}")
print(f"\n⏱️  {elapsed:.1f}s")
print()
print("📍 TRACE INSPECTION CHECKLIST for Run 1:")
print("    □ Find the trace named 'Run 1: focused stock query'")
print("    □ How many tool-call spans are there?")
print("    □ Which tools did this run call? Does it match the printout above?")
print("    □ What argument was passed, and what did the tool return?")
print("    □ What is the total token count?")

#### Run 2: A Broader Query

A broader question that may use multiple tools.

In [ ]:
print("🔬 RUN 2: Broad question that may use multiple tools")
print("=" * 60)

start = time.time()

result = await Runner.run(
    inventory_agent,
    input="Give me a full electronics report and flag anything out of stock.",
    run_config=RunConfig(workflow_name="Run 2: broad report query"),
)

elapsed = time.time() - start

print(f"Response: {result.final_output}")
print(f"🔧 Tools this run called: {tools_used(result)}")
print(f"\n⏱️  {elapsed:.1f}s")
print()
print("📍 TRACE INSPECTION CHECKLIST for Run 2:")
print("    □ Find the trace named 'Run 2: broad report query'")
print("    □ Which tools did this run call, and in what order?")
print("    □ Did it use get_category_report for the full report?")
print("    □ Did it rely on that alone, or also call check_stock?")
print("    □ Compare total tokens with Run 1: which used more, and why?")


### 💡 What These Traces Show

The printout shows which tools each run actually called.

In the dashboard, match that against the spans:

- how many model-call (LLM) spans fired

- which tools ran, and in what order

- the token difference vs Run 1

The exact tool count matters less than practicing how to read the trace structure.

---

## 🐛 Part 2: Debugging a Bad Run

The real value of tracing is diagnosing failures.

Here's an agent whose instructions steer it to the wrong tool.

Use the trace to see what it did, and why.

In [ ]:
# This agent has a deliberate instruction bug:
# It's told to use search_inventory for EVERYTHING,
# including questions where check_stock would give a precise answer.
# search_inventory returns broad matches without stock counts.
# check_stock returns exact counts.

buggy_instructions = (
    "You are an inventory assistant.\n"
    "For ALL questions, including specific stock counts, use search_inventory.\n"
    "Do not use check_stock or get_category_report under any circumstances."
)

buggy_agent = Agent(
    name="BuggyInventoryAgent",
    instructions=buggy_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Buggy agent ready")
print("    Instructions: use search_inventory for every request")
print("    Designed to produce imprecise answers for stock count questions")

#### Run Buggy Agent

In [ ]:
print("🐛 DEBUGGING DEMO: Broken Agent")
print("=" * 60)

# Ask for an exact stock count. Sound instructions would use check_stock,
# but the buggy instructions steer it to search_inventory instead.

broken_result = await Runner.run(
    buggy_agent,
    input="I need the exact stock count for laptops, how many units do we have?",
    run_config=RunConfig(workflow_name="Debug: buggy agent"),
)

print(f"Buggy agent response:")
print(broken_result.final_output)
print(f"🔧 Tools this run called: {tools_used(broken_result)}")

#### Run Correct Agent

In [ ]:
# Now run the correct agent on the same question for comparison

correct_result = await Runner.run(
    inventory_agent,
    input="I need the exact stock count for laptops, how many units do we have?",
    run_config=RunConfig(workflow_name="Debug: correct agent"),
)

print(f"Correct agent response:")
print(correct_result.final_output)
print(f"🔧 Tools this run called: {tools_used(correct_result)}")
print()
print("=" * 60)
print("📍 DEBUGGING CHECKLIST:")
print("    □ Find the traces named 'Debug: buggy agent' and 'Debug: correct agent'")
print("    □ Buggy trace: which tool did it call, and what did that tool return?")
print("    □ Correct trace: which tool(s) did it call? Does it include check_stock?")
print("    □ What difference in tool output explains the different responses?")
print()
print("    The fix: instruct the agent to use check_stock for exact counts.")
print("    That is how a trace takes you from symptom → evidence → candidate fix.")


In your own projects, open the trace for any failed eval case (Lesson 07) or guardrail violation first.

A trace narrows the likely cause by showing the recorded sequence of model, tool, guardrail, and handoff events.

Start with the tool path: was the problem in tool selection, arguments, output, or the instructions behind them?

Some failures also involve sessions, guardrails, external services, or model randomness.

---

## 📈 Part 3: Comparing Latency and Token Usage

Run the same query on `MODEL` and `REASONING_MODEL`.

Tokens are word or subword chunks used for pricing and context limits.

The trace shows the observed token usage and latency for each run.

Reusable pattern: hold the task constant, change one variable at a time (model, instructions, or tool set), then compare.

In [ ]:
# Same agent, same tools, same query, different model

analysis_instructions = (
    "You are an inventory analyst.\n"
    "Analyze inventory data and make restocking recommendations.\n"
    "Consider stock levels, prices, and categories in your analysis."
)

analysis_agent_mini = Agent(
    name="InventoryAnalyst_mini",
    instructions=analysis_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

analysis_agent_full = Agent(
    name="InventoryAnalyst_full",
    instructions=analysis_instructions,
    model=REASONING_MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# --------------------------------------------------------------
print("✅ Two analyst agents ready")
print(f"    Mini:  {MODEL}")
print(f"    Full:  {REASONING_MODEL}")

#### Same Query, Two Models

In [ ]:
analysis_query = (
    "Review all inventory categories. Identify which products need restocking "
    "and prioritize them by urgency. Explain your reasoning."
)

print("📈 MODEL COMPARISON")
print("=" * 60)
print(f"Query: {analysis_query}\n")

for label, agent in [(f"MODEL ({MODEL})", analysis_agent_mini),
                      (f"REASONING_MODEL ({REASONING_MODEL})", analysis_agent_full)]:
    start = time.time()

    result = await Runner.run(
        agent,
        input=analysis_query,
        run_config=RunConfig(workflow_name=f"Model comparison: {agent.name}"),
    )

    elapsed = time.time() - start

    print(f"\n--- {label} ---")
    print(f"Time: {elapsed:.1f}s")
    called_tools = tools_used(result)
    print(f"🔧 Tools this run called: {called_tools}")
    if not called_tools:
        print("    ⚠️  Recommendation was produced without inspecting inventory.")
    print(f"Response ({len(result.final_output)} chars):")
    print(result.final_output)

print("\n" + "=" * 60)
print("📍 COMPARISON CHECKLIST (find 'Model comparison: InventoryAnalyst_mini' and '..._full'):")
print("    □ Which model took longer in this run? By how much?")
print("    □ Which used more input tokens? Output tokens?")
print("    □ Did both runs call inventory tools, or did either recommend without checking stock?")
print("    □ Which produced the more actionable recommendation?")
print("    □ For this task, is the quality difference worth the token cost?")
print("    □ This is one comparison, not a benchmark: repeat before deciding.")

### 💡 Why This Matters

The trace shows observed token usage and span timing for these runs.

Combine token usage with current pricing to estimate model-token cost.

One run each is a single comparison, not a benchmark: repeat before choosing a model.

Compare recommendation quality only when both runs actually used the inventory tools.

In production, these same metrics support monitoring: watch for spikes against a baseline you track over time.

---

## 💪 Practice Exercises

### Exercise 1: Trace-Driven Debug

*Covers: tracing, diagnosing bugs from trace output*

A broken agent is provided. Run it, find the bug using only the trace, fix it, and confirm the fix.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Trace-Driven Debug
# --------------------------------------------------------------
# Objective: Use the dashboard trace to find and fix a bug.

# This agent has a bug that causes incorrect category reports.
# The bug is in its instructions, NOT in the tools.

category_broken_instructions = (
    "You are an inventory assistant.\n"
    "When asked about a category, use search_inventory with the category name.\n"
    "Do not use get_category_report."
)

category_agent_broken = Agent(
    name="CategoryAgent_broken",
    instructions=category_broken_instructions,
    model=MODEL,
    tools=[search_inventory, check_stock, get_category_report]
)

# TODO 1: Run this agent with:
#           "Give me a full report on the electronics category including total inventory value."
#           Set RunConfig(workflow_name="Exercise 1: category bug") so you can find the trace.

# TODO 2: Open the trace and answer:
#           a) Which tool did it call?
#           b) What did that tool return?
#           c) Did the response include the total inventory value?
#           d) Which tool would return that value?

# TODO 3: Create a fixed version of the agent with corrected instructions
#           Run the fixed agent with workflow_name="Exercise 1: category fixed"
#           Run the same query and check the trace:
#           - Which tool did the fixed agent call?
#           - Does the response now include the total inventory value?

# --- Write your code below this line ---

### Exercise 2: Your Own Latency Experiment

*Covers: tracing, designing your own comparison*

Pick one of three comparisons, run both conditions, and use the traces to answer the question you set.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Your Own Latency Experiment
# --------------------------------------------------------------
# Objective: Use traces to answer a question about agent performance.

# Choose ONE of these experiments:

# This is one observational comparison, not a benchmark: two runs cannot
# prove a general rule. To draw a firm conclusion, repeat each condition
# several times and compare the medians.

# Option A: Simple vs complex query
#     Run inventory_agent on a simple question and a complex question
#     Compare in the traces: tool-call count, total tokens, latency
#     Question: how did these two runs differ?

# Option B: 1 tool vs 3 tools in instructions
#     Create two agents: one with only check_stock, one with all 3 tools
#     Run both on: "How many laptops do we have?"
#     Compare in the traces: did tool count or latency differ between the two runs?

# Option C: Your own question
#     Design a comparison that uses traces to answer something you're curious about

# TODO 1: Choose an option and set up your comparison
# TODO 2: Run both conditions, giving each a distinct RunConfig(workflow_name=...)
#         so both traces are easy to find in the dashboard
# TODO 3: Compare the traces in the dashboard
# TODO 4: Write your findings as comments: what differed between the runs?

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Tracing is on by default:**

- A top-level `Runner.run()` creates a trace unless tracing is disabled, and a nested run joins the active trace

- The dashboard lists traces by workflow name, so set a descriptive one per run to find them

- A trace records a run's observable events (agent, model, tool, guardrail, and handoff spans), not the model's hidden reasoning
<br>
<br>

**Use traces to debug:**

- Start with the symptom (wrong output), then trace backward through tool calls

- Check tool arguments first: wrong input to a correct tool is a common failure

- Check tool outputs next: the tool may have returned something unexpected

- Check the model-call (LLM) span last: with sensitive-data capture on, inspect the recorded model input and response
<br>
<br>

**Use traces to monitor and decide:**

- Token counts are usage data: combine them with current pricing to estimate cost

- Per-step latency shows which step is the bottleneck in a run

- Unexpected tool-call patterns can point to instructions, tool descriptions, or model behavior

- Compare `MODEL` vs `REASONING_MODEL` on the same query, and repeat before committing to either

---

## 📍 Next Step

**Notebook 24: Capstone #3**  

Build a multi-tool agent with sessions, guardrails, human-in-the-loop approval, and tracing.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-23-tracing-and-observability)

---